# LLM Extraction Walkthrough

This notebook demonstrates using a local LLM (Phi-3.5-mini) to extract structured data from invoice images.

## Overview
1. **Model Setup** - Download and load the LLM
2. **OCR Extraction** - Get text from invoice (covered in previous notebook)
3. **Prompt Engineering** - Craft the extraction prompt
4. **LLM Inference** - Pass prompt to model and capture output
5. **Output Processing** - Parse and validate the response

## LLM Extraction Flow

```
┌─────────────────┐
│  Invoice Image  │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   OCR Extract   │  (pytesseract)
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   Raw OCR Text  │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  Build Prompt   │  (OCR text + JSON schema + instructions)
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│   LLM Model     │  (Phi-3.5-mini)
│   - temperature │  
│   - max_tokens  │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│  JSON Response  │
└────────┬────────┘
         │
         ▼
┌─────────────────┐
│ Structured Data │  (parsed & validated)
└─────────────────┘
```

## Step 1: Install Dependencies and Download Model

In [ ]:
!pip install llama-cpp-python pillow pytesseract pydantic -q

In [ ]:
# Download the Phi-3.5-mini model (~2.4GB)
from pathlib import Path
import urllib.request

MODEL_FILE = Path("./models/Phi-3.5-mini-instruct-Q4_K_M.gguf")
MODEL_URL = "https://huggingface.co/bartowski/Phi-3.5-mini-instruct-GGUF/resolve/main/Phi-3.5-mini-instruct-Q4_K_M.gguf"

MODEL_FILE.parent.mkdir(exist_ok=True)

if not MODEL_FILE.exists():
    print(f"Downloading model (~2.4GB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)
    print("✓ Download complete!")
else:
    print(f"✓ Model already exists")

## Step 2: Import Libraries

In [ ]:
import json
from pathlib import Path
from PIL import Image
import pytesseract
from llama_cpp import Llama
from IPython.display import display, Image as IPImage

## Step 3: Select Sample Invoice and Extract OCR Text

**Note:** OCR extraction was covered in the previous notebook. Here we'll quickly extract text to use with the LLM.

In [ ]:
# Select sample invoice
SAMPLE_IMAGE = "data/batch_1/batch_1/batch1_3/batch1-1022.jpg"

# Display the invoice
display(IPImage(filename=SAMPLE_IMAGE, width=600))

# Extract OCR text (covered in previous notebook)
image = Image.open(SAMPLE_IMAGE)
ocr_text = pytesseract.image_to_string(image)

print(f"\n✓ Extracted {len(ocr_text)} characters from invoice")
print(f"\nFirst 300 characters:\n{ocr_text[:300]}...")

## Step 4: Load the LLM Model

In [ ]:
# Load the LLM model
print("Loading LLM model (may take 10-30 seconds)...")

llm = Llama(
    model_path=str(MODEL_FILE),
    verbose=False
)

print("✓ Model loaded successfully!")

## Step 5: Define the JSON Schema

In [ ]:
# Define the JSON schema for invoice data extraction
invoice_schema = {
    "invoice_number": "string",
    "invoice_date": "string (YYYY-MM-DD format)",
    "vendor_name": "string",
    "vendor_address": "string",
    "customer_name": "string",
    "customer_address": "string",
    "total_amount": "number",
    "currency": "string",
    "line_items": [
        {
            "description": "string",
            "quantity": "number",
            "unit_price": "number",
            "total": "number"
        }
    ]
}

print("Invoice Data Schema:")
print(json.dumps(invoice_schema, indent=2))

## Step 6: Build the Extraction Prompt

The prompt combines:
1. Task instructions
2. OCR text from the invoice
3. JSON schema for output structure
4. Formatting rules

In [ ]:
def create_extraction_prompt(ocr_text, schema):
    """Build prompt for LLM to extract structured data"""
    schema_text = json.dumps(schema, indent=2)
    
    prompt = f"""You are a data extraction assistant. Extract structured information from the following invoice text.

Use the JSON schema provided below.
Populate all fields exactly as defined in that schema.
Do not add extra fields.
Do not rename fields.
Do not omit required fields.

Formatting rules:
- Return ONLY valid JSON.
- Do not include explanations, markdown, comments, or code blocks.
- If a field cannot be found, return null.
- All numeric values must be numbers, not strings.
- Convert dates to ISO format YYYY-MM-DD.

Invoice Text:
{ocr_text}

JSON Schema:
{schema_text}

JSON Output:"""
    
    return prompt

# Generate the prompt
prompt = create_extraction_prompt(ocr_text, invoice_schema)

print("EXTRACTION PROMPT")
print("=" * 80)
print(prompt[:500] + "...\n[truncated]")
print(f"\nFull prompt length: {len(prompt)} characters")

## Step 7: Pass Prompt to LLM and Capture Output

### Key Parameters:

- **`temperature`** (0.1): Controls randomness
  - 0.0-0.2 = deterministic (best for data extraction)
  - 0.5-0.7 = balanced
  - 0.8-1.0 = creative

- **`max_tokens`** (2048): Maximum response length
  - ~1500 words, sufficient for most invoices

In [ ]:
# Run LLM inference
print("Running LLM inference (may take 10-60 seconds)...\n")

response = llm(
    prompt,
    temperature=0.1,      # Low temperature for consistent output
    max_tokens=2048       # Max response length
)

print("✓ Inference complete!")

## Step 8: Examine the Response Structure

In [ ]:
# The response is a dictionary with metadata and the generated text
print(f"Response type: {type(response)}")
print(f"Response keys: {list(response.keys())}")
print(f"\nUsage stats:")
print(f"  Prompt tokens: {response['usage']['prompt_tokens']}")
print(f"  Completion tokens: {response['usage']['completion_tokens']}")
print(f"  Total tokens: {response['usage']['total_tokens']}")

## Step 9: Extract the Generated Text

In [ ]:
# Extract the generated text from response
generated_text = response["choices"][0]["text"].strip()

print("RAW LLM OUTPUT")
print("=" * 80)
print(generated_text)
print("=" * 80)

## Step 10: Parse JSON Output

Clean up markdown code blocks (if present) and parse the JSON.

In [ ]:
def extract_json_from_response(generated_text):
    """Extract and parse JSON from LLM response"""
    # Remove markdown code blocks if present
    if "```json" in generated_text:
        generated_text = generated_text.split("```json")[1].split("```")[0].strip()
    elif "```" in generated_text:
        generated_text = generated_text.split("```")[1].split("```")[0].strip()
    
    try:
        return json.loads(generated_text)
    except json.JSONDecodeError as e:
        print(f"JSON parsing error: {e}")
        return None

# Parse the JSON
structured_data = extract_json_from_response(generated_text)

if structured_data:
    print("✓ JSON parsed successfully!\n")
    print("STRUCTURED INVOICE DATA")
    print("=" * 80)
    print(json.dumps(structured_data, indent=2))
    print("=" * 80)
else:
    print("✗ Failed to parse JSON")

## Step 11: Compare Input vs Output

See the transformation from messy OCR text to clean structured data.

In [ ]:
print("INPUT: Raw OCR Text (first 300 chars)")
print("-" * 80)
print(ocr_text[:300] + "...\n")

print("OUTPUT: Structured JSON Data")
print("-" * 80)
if structured_data:
    print(json.dumps(structured_data, indent=2))

print("\n✓ Transformation complete: Unstructured → Structured")

---

## 🎯 Your Turn: Experiment with LLM Parameters

Now it's time to experiment! Try the following exercises to understand how different parameters affect the output.

### Exercise 1: Temperature Experiment

Run the extraction with different temperature values and observe the differences.

**Task:** Complete the code below to extract data with three different temperatures.

In [ ]:
# TODO: Run extraction with different temperatures
temperatures = [0.0, 0.5, 1.0]

for temp in temperatures:
    print(f"\n{'='*80}")
    print(f"TEMPERATURE: {temp}")
    print(f"{'='*80}")
    
    # TODO: Call the LLM with the current temperature
    # response = llm(prompt, temperature=temp, max_tokens=2048)
    
    # TODO: Extract and parse the JSON
    # generated_text = response["choices"][0]["text"].strip()
    # data = extract_json_from_response(generated_text)
    
    # TODO: Print the vendor_name and total_amount from the extracted data
    # if data:
    #     print(f"Vendor: {data.get('vendor_name')}")
    #     print(f"Total: {data.get('total_amount')}")
    
    pass  # Remove this line when you add your code

# Question: Which temperature gave the most consistent results?

#### Solution (Don't peek until you've tried!)

In [ ]:
# Solution: Temperature Experiment
temperatures = [0.0, 0.5, 1.0]

for temp in temperatures:
    print(f"\n{'='*80}")
    print(f"TEMPERATURE: {temp}")
    print(f"{'='*80}")
    
    # Call the LLM with the current temperature
    response = llm(prompt, temperature=temp, max_tokens=2048)
    
    # Extract and parse the JSON
    generated_text = response["choices"][0]["text"].strip()
    data = extract_json_from_response(generated_text)
    
    # Print key fields from the extracted data
    if data:
        print(f"Vendor: {data.get('vendor_name')}")
        print(f"Total: {data.get('total_amount')}")
        print(f"Invoice #: {data.get('invoice_number')}")
    else:
        print("Failed to extract data")

# Answer: Temperature 0.0 typically gives the most consistent, deterministic results
# Higher temperatures (0.5, 1.0) may vary between runs

### Exercise 2: Process a Different Invoice

**Task:** Pick a different invoice from the `data/batch_1` folder and extract its data.

In [ ]:
# TODO: Choose a different invoice image
# new_image = "data/batch_1/batch_1/batch1_3/batch1-1026.jpg"  # Change this path

# TODO: Extract OCR text from the new image
# new_image_obj = Image.open(new_image)
# new_ocr_text = pytesseract.image_to_string(new_image_obj)

# TODO: Create a new prompt with the new OCR text
# new_prompt = create_extraction_prompt(new_ocr_text, invoice_schema)

# TODO: Run LLM inference
# new_response = llm(new_prompt, temperature=0.1, max_tokens=2048)

# TODO: Extract and display the structured data
# new_generated = new_response["choices"][0]["text"].strip()
# new_data = extract_json_from_response(new_generated)
# print(json.dumps(new_data, indent=2))

pass  # Remove this line when you add your code

#### Solution (Don't peek until you've tried!)

In [ ]:
# Solution: Process a Different Invoice

# Choose a different invoice image
new_image = "data/batch_1/batch_1/batch1_3/batch1-1026.jpg"

# Display the new invoice
print(f"Processing: {new_image}")
display(IPImage(filename=new_image, width=600))

# Extract OCR text from the new image
new_image_obj = Image.open(new_image)
new_ocr_text = pytesseract.image_to_string(new_image_obj)
print(f"\nExtracted {len(new_ocr_text)} characters")

# Create a new prompt with the new OCR text
new_prompt = create_extraction_prompt(new_ocr_text, invoice_schema)

# Run LLM inference
print("\nRunning LLM inference...")
new_response = llm(new_prompt, temperature=0.1, max_tokens=2048)

# Extract and display the structured data
new_generated = new_response["choices"][0]["text"].strip()
new_data = extract_json_from_response(new_generated)

if new_data:
    print("\n✓ Extraction successful!")
    print("\nSTRUCTURED DATA:")
    print(json.dumps(new_data, indent=2))
else:
    print("\n✗ Extraction failed")

---

## 📝 Reflection Questions

After completing the exercises, consider:

1. **Temperature Impact**: How did different temperature values affect the consistency and accuracy of extraction?

2. **Prompt Engineering**: What happens when you add more specific instructions to the prompt? Does it improve accuracy?

3. **Error Handling**: Why is retry logic important for production systems?

4. **Performance**: How long does it take to process one invoice? How could you optimize for batch processing?

5. **Limitations**: What types of invoices or fields might be challenging for this approach?